# ConjunctNet

In [ ]:
import os
import glob

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from PIL import Image
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
import pandas as pd

data_dir = 'link/to/data/'
save_path = 'link/to/save/path/'
fig_path = save_path + 'figures/'
model_dir = save_path + 'models/'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

## Masking

In [ ]:
# image masking - only done once
def bbox2(img):
    # from https://stackoverflow.com/a/31402351/19249364
    rows = np.any(img, axis=1)
    cols = np.any(img, axis=0)
    ymin, ymax = np.where(rows)[0][[0, -1]]
    xmin, xmax = np.where(cols)[0][[0, -1]]
    return ymin, ymax, xmin, xmax

for _, row in images_df.iterrows():
    img_path = row['image_path']
    mask_path = row['mask_path']
    img_name = row['image_name']
    imgid = row['imgid']
    dest_dir = data_dir+'masked/'+imgid+'/'
    if (not os.path.exists(mask_path)):
        print(img_name+" mask not found")
        continue
    if os.path.exists(dest_dir+img_name):
        continue
    img = cv2.imread(img_path, cv2.IMREAD_COLOR)
    cv_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    _, mask = cv2.threshold(cv_mask, 240, 255, cv2.THRESH_BINARY)
    # find the bounding box of the mask
    ymin, ymax, xmin, xmax = bbox2(mask)
    # crop img
    img = img[ymin:ymax, xmin:xmax]
    mask = mask[ymin:ymax, xmin:xmax]
    img = np.array(img)
    mask = np.array(mask)
    img = cv2.bitwise_and(img, img, mask=mask)
    os.makedirs(dest_dir, exist_ok=True)
    cv2.imwrite(dest_dir+img_name, img)

## Dataset Utilities

In [ ]:
class MetaMaskDataset(Dataset):
    def __init__(self, dataframe, transform=None, label_col='label', augment=None):
        self.data_frame = dataframe
        self.transform = transform
        masked_paths = list(dataframe['masked_path'])
        labels = list(dataframe[label_col].astype(int))
        self.locc = []
        self.cached_images = []
        self.labels = []
        self.masked_paths = []
        self.augment = augment
        metadata = np.array(dataframe.drop(columns=[label_col]))
        self.meta_keys = dataframe.drop(columns=[label_col]).columns
        self.meta_key_to_id = {key: i for i, key in enumerate(self.meta_keys)}
        
        self.metadata = []
        for img_path in masked_paths:
            img = Image.open(img_path).convert("RGB")
            if self.transform:
                img = self.transform(img)                    
            self.masked_paths.append(img_path)
            self.cached_images.append(img)
            self.labels.append(labels[masked_paths.index(img_path)])
            self.metadata.append([
                *metadata[masked_paths.index(img_path)],
                ])

    def __len__(self):
        return len(self.masked_paths)
    
    def get_meta_key(self, id):
        return self.meta_keys[id]
    
    def get_meta_id(self, key):
        return self.meta_key_to_id[key]

    def __getitem__(self, idx):
        if self.augment:
            augmented_img = self.augment(self.cached_images[idx])
            return (augmented_img, self.metadata[idx], self.labels[idx])
        return (self.cached_images[idx], self.metadata[idx], self.labels[idx])

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
    Clahe(clip_limit=1.5)
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224), antialias=True),
    Clahe(clip_limit=1.5),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0),
    transforms.RandomAdjustSharpness(2, p=1),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
TASK = 'pterygium'
NUM_FOLDS = 5
train_dfs = []
test_dfs = []
for fold in range(NUM_FOLDS):
    train_df = pd.read_csv(splits_dir+f'{TASK}_folds/train_fold_{fold}.csv', dtype={'imgid': str})
    test_df = pd.read_csv(splits_dir+f'{TASK}_folds/test_fold_{fold}.csv', dtype={'imgid': str})
    train_dfs.append(train_df)
    test_dfs.append(test_df)

## Training Utilities

In [ ]:
def calculate_accuracy(loader, model, criterion):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for (images, _, labels) in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images)
            targets = F.one_hot(labels.to(torch.int64), num_classes=2).float()
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            predicted = torch.argmax(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100. * correct / total
    average_loss = total_loss / len(loader)
    return accuracy, average_loss

In [ ]:
def train_model(model, train_loader, val_loader, criterion, 
                    optimizer, scheduler=None, experiment_name="experiment", 
                    num_epochs=40, label_name='conjunctiva',
                    save_last=False, verbose=True
                    ):
    train_accuracies = []
    val_accuracies = []
    train_losses = []
    val_losses = []
    best_val_acc = 0.0
    checkpoint_val_loss = 1000.0
    train_step = 0

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        total = 0
        correct = 0
        for i, (images, _, labels) in enumerate(train_loader):
            optimizer.zero_grad()
            images, labels = images.to(device), labels.float().to(device)
            outputs = model(images)
            targets = F.one_hot(labels.to(torch.int64), num_classes=2).float()
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            if scheduler is not None:
                scheduler.step()
            running_loss += loss.item()
            predicted = torch.argmax(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            train_step += 1
            torch.cuda.empty_cache()
        running_loss /= len(train_loader)
        train_accuracy = 100. * correct / total
        
        val_accuracy, val_loss = calculate_accuracy(val_loader, model, criterion)

        if verbose and verbose!='final':
            print(f'Epoch {epoch + 1}, Training Accuracy: {train_accuracy:.2f}%, Training Loss: {running_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%, Validation Loss: {val_loss:.4f}')
        train_accuracies.append(train_accuracy)
        val_accuracies.append(val_accuracy)
        train_losses.append(running_loss)
        val_losses.append(val_loss)
        if val_loss <= checkpoint_val_loss:
            if val_loss == checkpoint_val_loss and val_accuracy > best_val_acc:
                continue
            best_val_acc = val_accuracy
            checkpoint_val_loss = val_loss
            torch.save(model.state_dict(), model_dir + f'{label_name}-{experiment_name}_best.pth')
    if save_last:
        torch.save(model.state_dict(), model_dir + f'{label_name}-{experiment_name}.pth')

    if verbose:
        print('Finished Training')
        print(f'Checkpoint Validation Accuracy: {best_val_acc:.2f}%')
        print(f'Checkpoint Validation Loss: {checkpoint_val_loss:.4f}')
    return model, train_accuracies, val_accuracies, train_losses, val_losses

In [ ]:
def train_evaluate(model_class, label_name, model_name, train_loader, test_loader, 
        train_epochs=40, tune_epochs=10, train_lr=0.01, tune_lr=0.0001,
        save_weights=True, verbose='final'):
    results_df = pd.DataFrame(columns=[
        'label_name', 'model_name', 'test_loss', 'test_acc', 
        'test_auc', 'test_f1', 'test_precision', 'test_sensitivity', 'test_specificity'
        ])
    num_epochs = train_epochs
    scheduler = None

    model = model_class(num_classes=2)
    model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.SGD(model.parameters(), lr=train_lr, momentum=0.9, weight_decay=10e-5)
    model, _, _, train_losses_partial, val_losses_partial = train_model(
                                        model, train_loader, test_loader,
                                        criterion, optimizer, scheduler=scheduler,
                                        experiment_name=model_name,
                                        num_epochs=num_epochs,
                                        label_name=label_name, save_last=False,
                                        verbose=verbose
                                        )
    model.load_state_dict(torch.load(f"models/{label_name}-{model_name}_best.pth"))
    for param in model.parameters():
        param.requires_grad = True
    optimizer = optim.SGD(model.parameters(), lr=tune_lr, momentum=0.9, weight_decay=10e-5)
    model, _, _, train_losses_tuned, val_losses_tuned = train_model(
                                        model, train_loader, test_loader,
                                        criterion, optimizer, scheduler=scheduler,
                                        experiment_name=f'{model_name}-tuned',
                                        num_epochs=tune_epochs,
                                        label_name=label_name, save_last=False,
                                        verbose=verbose
                                        )
    # load the best weights depending on the validation loss 
    if min(val_losses_partial) < min(val_losses_tuned):
        best_test_loss = min(val_losses_partial)
        checkpoint_train_loss = train_losses_partial[val_losses_partial.index(best_test_loss)]
        model.load_state_dict(torch.load(f"models/{label_name}-{model_name}_best.pth", weights_only=False))
        os.remove(f"models/{label_name}-{model_name}-tuned_best.pth")
    else:
        best_test_loss = min(val_losses_tuned)
        checkpoint_train_loss = train_losses_tuned[val_losses_tuned.index(best_test_loss)]
        model.load_state_dict(torch.load(f"models/{label_name}-{model_name}-tuned_best.pth", weights_only=False))
        os.remove(f"models/{label_name}-{model_name}_best.pth")
        # rename tuned to base
        os.rename(f"models/{label_name}-{model_name}-tuned_best.pth", 
                    f"models/{label_name}-{model_name}_best.pth")
    test_TP, test_TN, test_FP, test_FN = calculate_confusion_matrix(model, test_loader)
    test_accuracy, test_precision, test_recall, test_f1, test_sensitivity, test_specificity = calculate_metrics(test_TP, test_TN, test_FP, test_FN, verbose=True)
    test_auc = calculate_auc(model, test_loader)
    results_df = pd.concat([results_df, pd.DataFrame([[
        label_name, model_name, best_test_loss, test_accuracy, test_auc, test_f1, test_precision, test_sensitivity, test_specificity
    ]], columns=results_df.columns)])
    if not save_weights:
        os.remove(f"models/{label_name}-{model_name}-tuned_best.pth")
    print("\t\t\t<><><>")
    if draw_figs:
        train_confmat = np.array([[TN, FP], [FN, TP]])
        test_confmat = np.array([[test_TN, test_FP], [test_FN, test_TP]])
        fig = plt.figure(figsize=(12, 5))
        ax1 = fig.add_subplot(121)
        ax3 = fig.add_subplot(122)
        draw_confusion_matrix(train_confmat, ax=ax1, subtitle='Train')
        draw_confusion_matrix(test_confmat, ax=ax3, subtitle='Test')
        plt.suptitle(f'{model_name}\n{label_name}', fontsize=14)
        plot_roc_curves(model, train_loader, None, test_loader, title=f'ROC Curve ({model_name})\n{label_name}')
    results_df.to_csv(f'results/{label_name}-{model_name}_results.csv', index=False)
    del model, optimizer
    torch.cuda.empty_cache()
    return results_df

In [ ]:
from sklearn.metrics import roc_curve, auc

def calculate_fpr_tpr(model, dataloader):
    y_true = []
    y_pred = []
    model.eval()
    if metadata_labels is not None and metadata_ids is None:
        metadata_ids = [dataloader.dataset.get_meta_id(key) for key in metadata_labels]
    with torch.no_grad():
        for (images, metadata, targets) in dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            predicted = outputs[:, 1]
            y_true.extend(targets.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    fpr, tpr, thresholds = roc_curve(y_true, y_pred)
    return fpr, tpr, thresholds

def calculate_auc(model, dataloader, use_metadata=False, metadata_labels=None, metadata_ids=None):
    fpr, tpr, thresholds = calculate_fpr_tpr(model, dataloader)
    return auc(fpr, tpr)

def get_roc_curve(model, dataloader):
    fpr, tpr, thresholds = calculate_fpr_tpr(model, dataloader)
    auc_score = auc(fpr, tpr)
    return fpr, tpr, thresholds, auc_score

def plot_roc_curve(model, dataloader,
                        use_metadata=False, metadata_labels=None, metadata_ids=None,
                        title="ROC Curve", color="darkorange", show_fig=False):
    fpr, tpr, thresholds = calculate_fpr_tpr(model, dataloader)
    auc_score = auc(fpr, tpr)
    fig = plt.figure()
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{title}, AUC = {auc_score:.2f}')
    plt.plot([0, 1], [0, 1], color='black', lw=1, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('1-Specificity')
    plt.ylabel('Sensitivity')
    plt.legend(loc="lower right")
    if show_fig:
        plt.show()
    return fpr, tpr, thresholds, auc_score, fig

def plot_roc_curves(model, dataloader, valloader, testloader, title="Receiver Operating Characteristic", 
        use_metadata=False, metadata_labels=None, metadata_ids=None):
    fpr, tpr, threshold = calculate_fpr_tpr(model, dataloader)
    train_roc_auc = auc(fpr, tpr)
    if valloader is not None:
        val_fpr, val_tpr, val_threshold = calculate_fpr_tpr(model, valloader)
        val_roc_auc = auc(val_fpr, val_tpr)
    else:
        val_roc_auc = None
    test_fpr, test_tpr, test_threshold = calculate_fpr_tpr(model, testloader
    test_roc_auc = auc(test_fpr, test_tpr)
    fig = plt.figure()
    plt.plot(fpr, tpr, color='darkorange', lw=2, label='Train (auc = %0.2f)' % train_roc_auc)
    if valloader is not None:
        plt.plot(val_fpr, val_tpr, color='green', lw=2, label='Validation (auc = %0.2f)' % val_roc_auc)
    plt.plot(test_fpr, test_tpr, color='red', lw=2, label='Test (auc = %0.2f)' % test_roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title)
    plt.legend(loc="lower right")
    plt.show()
    return train_roc_auc, val_roc_auc, test_roc_auc


In [ ]:
def calculate_confusion_matrix(model, dataloader):
    model.eval()
    TP, TN, FP, FN = 0, 0, 0, 0
    with torch.no_grad():
        for (images, metadata, labels) in dataloader:
            images, labels = images.to(device), labels.float().to(device)
            outputs = model(images)
            outputs = outputs.squeeze()
            # patch buggy behaviour on the last batch of dataloaders
            if(labels.shape[0]==1):
                outputs = outputs.unsqueeze(0)
            predicted = torch.argmax(outputs, 1)
            TP += ((predicted == 1) & (labels == 1)).sum().item()
            TN += ((predicted == 0) & (labels == 0)).sum().item()
            FP += ((predicted == 1) & (labels == 0)).sum().item()
            FN += ((predicted == 0) & (labels == 1)).sum().item()
    return TP, TN, FP, FN
    
def calculate_metrics(TP, TN, FP, FN, verbose=False):
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    if TP + FP == 0:
        precision = 0
    else:
        precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * (precision * recall) / (precision + recall)
    sensitivity = TP / (TP + FN)
    specificity = TN / (TN + FP)
    if verbose:
        print(f"{'Metric':<12}{'Score':<8}")
        print(f"{'Accuracy':<12}{accuracy:.4f}")
        print(f"{'Precision':<12}{precision:.4f}")
        print(f"{'Recall':<12}{recall:.4f}")
        print(f"{'F1':<12}{f1:.4f}")
        print(f"{'Sensitivity':<12}{sensitivity:.4f}")
        print(f"{'Specificity':<12}{specificity:.4f}")
    return accuracy, precision, recall, f1, sensitivity, specificity

def draw_confusion_matrix(conf_matrix, ax=None, subtitle=None, class_names=['Negative', 'Positive']):
    if ax is None:
        ax = plt.gca()
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(f'{"" if subtitle is None else subtitle}')
    return ax

In [ ]:
def kfold_cross_validation(model_class, label_name, 
        model_name, train_dfs, test_dfs, train_transform, test_transform, augment=None, 
        train_epochs=40, tune_epochs=10, train_lr=0.01, tune_lr=0.0001, dataset_class=MetaMaskDataset, batch_size=16,
        save_weights=True, label_col='conjunctiva', verbose='final',
        num_folds=5
        ):
    results_df = pd.DataFrame(columns=[
        'fold', 'test_loss', 'test_acc', 
        'test_auc', 'test_f1', 'test_precision', 
        'test_sensitivity', 'test_specificity'
    ])
    num_epochs = train_epochs
    for fold in fold_range:
        print(f"Fold: {fold}")
        train_df = train_dfs[fold]
        test_df = test_dfs[fold]
        trainset = dataset_class(train_df, transform=transform, label_col=label_col, augment=augment)
        testset = dataset_class(test_df, transform=test_transform, label_col=label_col)
        train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(testset, batch_size=batch_size, shuffle=False)
        
        model = model_class(num_classes=2)
        model.to(device)
        criterion = nn.BCELoss()
        optimizer = optim.SGD(model.parameters(), lr=train_lr, momentum=0.9, weight_decay=10e-5)
        model, _, _, train_losses_partial, val_losses_partial = train_model(
                                            model, train_loader, test_loader,
                                            criterion, optimizer, scheduler=scheduler,
                                            experiment_name=f'{model_name}_f{fold}',
                                            num_epochs=num_epochs,
                                            label_name=label_name, save_last=False,
                                            verbose=verbose
                                            )
        print("Tuning model...")
        model.load_state_dict(torch.load(f"models/{label_name}-{model_name}_f{fold}_best.pth", weights_only=False))
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.SGD(model.parameters(), lr=tune_lr, momentum=0.9, weight_decay=10e-5)
        model, _, _, train_losses_tuned, val_losses_tuned = train_model(
                                            model, train_loader, test_loader,
                                            criterion, optimizer, scheduler=scheduler,
                                            experiment_name=f'{model_name}-tuned_f{fold}',
                                            num_epochs=tune_epochs,
                                            label_name=label_name, save_last=False,
                                            verbose=verbose
                                            )
        # load the best weights depending on the validation loss 
        if min(val_losses_partial) < min(val_losses_tuned):
            print("Checkpoint on partial model.")
            best_test_loss = min(val_losses_partial)
            checkpoint_train_loss = train_losses_partial[val_losses_partial.index(best_test_loss)]
            model.load_state_dict(torch.load(f"models/{label_name}-{model_name}_f{fold}_best.pth", weights_only=False))
            os.remove(f"models/{label_name}-{model_name}-tuned_f{fold}_best.pth")
            model_path = f"models/{label_name}-{model_name}_f{fold}_best.pth"
        else:
            print("Checkpoint on tuned model.")
            best_test_loss = min(val_losses_tuned)
            checkpoint_train_loss = train_losses_tuned[val_losses_tuned.index(best_test_loss)]
            model.load_state_dict(torch.load(f"models/{label_name}-{model_name}-tuned_f{fold}_best.pth", weights_only=False))
            os.remove(f"models/{label_name}-{model_name}_f{fold}_best.pth")
            os.rename(f"models/{label_name}-{model_name}-tuned_f{fold}_best.pth", 
                        f"models/{label_name}-{model_name}_f{fold}_best.pth")
            model_path = f"models/{label_name}-{model_name}_f{fold}_best.pth"
        test_TP, test_TN, test_FP, test_FN = calculate_confusion_matrix(model, test_loader)
        test_accuracy, test_precision, test_recall, test_f1, test_sensitivity, test_specificity = calculate_metrics(test_TP, test_TN, test_FP, test_FN, verbose=True)
        test_auc = calculate_auc(model, test_loader)
        results_df = pd.concat([results_df, pd.DataFrame([[
            fold, best_test_loss, test_accuracy, test_auc, test_f1, test_precision, test_sensitivity, test_specificity
        ]], columns=results_df.columns)])
        results_df.to_csv(f'results/{label_name}-{model_name}_results_partial.csv', index=False)
        del model, optimizer, train_loader, test_loader, trainset, testset
        torch.cuda.empty_cache()
        if not save_weights:
            os.remove(model_path)            
        print("\t\t\t<><><>")

    results_df.to_csv(f'results/{label_name}-{model_name}_results.csv', index=False)
    if os.path.exists(f'results/{label_name}-{model_name}_results_partial.csv'):
        os.remove(f'results/{label_name}-{model_name}_results_partial.csv')

    return results_df


In [ ]:
# from https://scikit-learn.org/1.5/auto_examples/model_selection/plot_roc_crossval.html
def kfold_aucs(test_dfs, num_folds, 
                        model_name, model_class, label_name='pterygium',
                        transform=None, augment=None, batch_size=8,
                        dataset_class=MetaMaskDataset, label_col='conjunctiva'
                        ):
    fprs = []
    tprs = []
    auc_scores = []
    mean_fpr = np.linspace(0, 1, 100)
    for fold in range(num_folds):
        model = model_class(num_classes=2)
        if os.path.exists(f'models/{label_name}-{model_name}_f{fold}_best.pth'):
            model.load_state_dict(torch.load(f'models/{label_name}-{model_name}_f{fold}_best.pth', weights_only=False))
        else:
            model.load_state_dict(torch.load(f'models/{label_name}-{model_name}-tuned_f{fold}_best.pth', weights_only=False))
        model.to(device)
        model.eval()
        test_set = dataset_class(test_dfs[fold], transform=transform, augment=augment, label_col=label_col)
        test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)
        fpr, tpr, _, auc_score = get_roc_curve(model, test_loader)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        auc_scores.append(auc_score)
    return mean_fpr, tprs, auc_scores

## Model Definitions

In [ ]:
from torchvision import models
from torchvision.models._api import WeightsEnum
from torch.hub import load_state_dict_from_url

In [ ]:
class ConjunctAlexNet(nn.Module):
    def __init__(self, num_classes=2):
        super(ConjunctAlexNet, self).__init__()
        alexnet = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        modules = list(alexnet.children())[:-1]
        self.alexnet = nn.Sequential(*modules)
        for param in self.alexnet.parameters():
            param.requires_grad = False
        for param in list(self.alexnet.children())[-2][-3:].parameters():
            param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(9216, 1028),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(1028, 128),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        for param in self.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        x = self.alexnet(x)
        x = self.classifier(x.view(x.size(0), -1))
        return x

In [ ]:
class ConjunctDenseNet121(nn.Module):
    def __init__(self, num_classes=2):
        super(ConjunctDenseNet121, self).__init__()
        densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
        self.densenet = densenet
        for param in self.densenet.parameters():
            param.requires_grad = False
        for param in list(self.densenet.children())[:-1][-1][-2].denselayer16.parameters():
            param.requires_grad = True
        self.densenet.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1024, 128),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        for param in self.densenet.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        x = self.densenet(x)
        return x

In [ ]:
class ConjunctEfficientNetB0(nn.Module):
    def __init__(self, num_classes=2, meta_len=2):
        super(ConjunctEfficientNetB0, self).__init__()
        self.effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        modules = list(self.effnet.children())[:-1]
        self.effnet = nn.Sequential(*modules)
        for param in self.effnet.parameters():
            param.requires_grad = False
        for param in list(self.effnet.children())[-2][-2:].parameters():
            param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 128),
            nn.ReLU(),  
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        
    def forward(self, x):
        x = self.effnet(x)
        x = self.classifier(x.view(x.size(0), -1))
        return x

In [ ]:
class ConjunctEfficientNetB3(nn.Module):
    def __init__(self, num_classes=2, meta_len=2):
        super(ConjunctEfficientNetB3, self).__init__()
        self.effnet = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        modules = list(self.effnet.children())[:-1]
        self.effnet = nn.Sequential(*modules)
        for param in self.effnet.parameters():
            param.requires_grad = False
        for param in list(self.effnet.children())[-2][-2:].parameters():
            param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1536, 128),
            nn.ReLU(),  
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        
    def forward(self, x):
        x = self.effnet(x)
        x = self.classifier(x.view(x.size(0), -1))
        return x

In [ ]:
class ConjunctMobileNetV2(nn.Module):
    def __init__(self, num_classes=2):
        super(ConjunctMobileNetV2, self).__init__()
        self.mobilenet_v2 = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        for param in self.mobilenet_v2.parameters():
            param.requires_grad = False
        for param in list(self.mobilenet_v2.children())[-2][-1].parameters():
            param.requires_grad = True
        self.mobilenet_v2.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(1280, 128),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        for param in self.mobilenet_v2.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        x = self.mobilenet_v2(x)
        return x

In [ ]:
class ConjunctResNet101(nn.Module):
    def __init__(self, num_classes=2):
        super(ConjunctResNet101, self).__init__()
        resnet = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        for param in self.resnet.parameters():
            param.requires_grad = False
        for param in list(self.resnet.children())[-1].parameters():
            param.requires_grad = True
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(2048, 128),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        for param in self.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        x = self.resnet(x)
        x = self.classifier(x.view(x.size(0), -1))
        return x


In [ ]:
class ConjunctVGG16(nn.Module):
    def __init__(self, num_classes=2):
        super(ConjunctVGG16, self).__init__()
        self.vgg16 = models.vgg16_bn(weights=models.VGG16_BN_Weights.DEFAULT)
        for param in self.vgg16.parameters():
            param.requires_grad = False
        for param in list(self.vgg16.children())[-3][-3:].parameters():
            param.requires_grad = True
        self.vgg16.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(25088, 128),
            nn.ReLU(), 
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
            nn.Softmax(dim=1)
        )
        for param in self.vgg16.classifier.parameters():
            param.requires_grad = True
    
    def forward(self, x):
        x = self.vgg16(x)
        return x

### Calling Training & Evaluation

In [ ]:
results_df = kfold_cross_validation(
    ConjunctEfficientNetB0,
    'screening', 'efficientnet-b0',
    train_dfs, test_dfs, transform, test_transform, augment=augment,
    train_epochs=40, tune_epochs=10,
    train_lr=0.01, tune_lr=0.0001,
    dataset_class=MetaMaskDataset
    )
results_df

## Saliency Map Generation

In [ ]:
# library: https://github.com/jacobgil/pytorch-grad-cam
from pytorch_grad_cam import GradCAMPlusPlus, GradCAM, FullGrad, HiResCAM, EigenCAM, XGradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

def gradcam_fold(data_df, fold, 
                        model_class, model_name, target_layers, label_name, experiment_name,
                        transform=None, augment=None, 
                        dataset_class=MetaMaskDataset, label_col='conjunctiva',
                        dataset_type='internal'
                        ):
    data_set = dataset_class(data_df, transform=transform, augment=augment, label_col=label_col)
    data_loader = DataLoader(data_set, batch_size=8, shuffle=False)
    model = model_class(num_classes=2).to(device)
    model.eval()
    if os.path.exists(f'models/{label_name}-{model_name}_f{fold}_best.pth'):
        model.load_state_dict(torch.load(f'models/{label_name}-{model_name}_f{fold}_best.pth', weights_only=False))
    else:
        print(f"Model {label_name}-{model_name}_f{fold}_best.pth does not exist.")
        return

    cams = [
        GradCAMPlusPlus(model=model, target_layers=target_layers),
    ]
    cam_names = [
        'gpp', 
    ]

    imgid_idx = data_loader.dataset.get_meta_id('imgid')
    image_path_idx = data_loader.dataset.get_meta_id('image_path')
    image_name_idx = data_loader.dataset.get_meta_id('image_name')
    masked_path_idx = data_loader.dataset.get_meta_id('masked_path')

    for i, (images, metadata, labels) in enumerate(data_loader):
        images = images.to(device)
        labels = labels.to(device)
        imgid = metadata[imgid_idx]
        outputs = model(images)
        cam_batches = []
        for cam in cams:
            cam_batch = cam(input_tensor=images)
            cam_batches.append(cam_batch)
        predicted = torch.argmax(outputs, 1).cpu().numpy()
        for j in range(len(images)):
            img = images[j].cpu().data.numpy()
            img = np.transpose(img, (1, 2, 0))  # CHW to HWC
            og_img = cv2.imread(metadata[masked_path_idx][j])
            og_img = cv2.cvtColor(og_img, cv2.COLOR_BGR2RGB)
            og_img = (og_img - og_img.min()) / (og_img.max() - og_img.min())
            filename = metadata[image_name_idx][j].split('.')[0]
            plt.imsave(os.path.join(save_dir, f"t{labels[j]}_p{predicted[j]}_{filename}.jpg"), og_img)  
            for k, cam in enumerate(cams):
                grayscale_cam = cam_batches[k][j, :]
                grayscale_cam = cv2.resize(grayscale_cam, (og_img.shape[1], og_img.shape[0]))
                visualization = show_cam_on_image(og_img, grayscale_cam, use_rgb=True)
                plt.imsave(os.path.join(save_dir, f"t{labels[j]}_p{predicted[j]}_{filename}_{cam_names[k]}.jpg"), visualization)